# 00-Setup: 10분 사전 점검

이 노트북은 본 수업(`01`~`06`)을 시작하기 전에 미리 끝내 두는 **10분 환경 점검**입니다. 내 컴퓨터에서 `rag_lab` 패키지, 한국어 형태소 분석기(Kiwi), 로컬 임베딩 모델, 재순위(reranker) 모델, Deep Agents, 그리고 실제 외부 API 호출이 정상적으로 동작하는지 위에서 아래로 차례대로 확인합니다.

## 학습 목표

- 패키지 설치는 저장소 루트에서 한 번만 하고, 노트북 실행은 `Day02_new/` 폴더에서 해야 하는 이유를 설명할 수 있습니다.
- 실제 모델을 호출하려면 `Day02_new/.env`에 `ANTHROPIC_API_KEY`와 판정 API 키(`TYPESAFE_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` 중 하나)가 모두 필요하며, 키가 빠졌을 때 어느 단계에서 멈추는지 확인할 수 있습니다.
- 내 실습 환경에서 `rag_lab`, Kiwi, 로컬 임베딩 모델, 재순위 모델, Deep Agents, 실제 모델 호출이 모두 정상 동작하는지 직접 눈으로 확인합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 설치와 폴더 구조 확인 | 2 |
| 2 | 키 상태 확인, `.env` 설정, 판정 API 호출 | 3 |
| 3 | 토크나이저, 로컬 임베딩 모델, 재순위 모델 점검 | 2 |
| 4 | 실제 모델 호출과 Deep Agents 한 턴 실행 | 2 |
| 5 | 이후 수업 시간표와 비용 규칙 안내 | 1 |

## 호출 규모

이 노트북에서 외부 유료 API를 호출하는 셀은 총 세 곳입니다. 2절의 판정 API 확인(1회), 4절의 모델 직접 호출(1회)과 Deep Agents 한 턴 실행(1회)을 합쳐 **총 3회 내외**입니다(Deep Agents가 내부에서 추가 호출을 하면 횟수가 조금 늘어날 수 있습니다). 임베딩과 재순위 모델은 내 컴퓨터(CPU)에서 직접 실행되므로 별도의 API 호출이나 비용이 발생하지 않습니다. 만약 필수 API 키가 설정되어 있지 않다면 2절의 확인 셀에서 한국어 안내 메시지와 함께 즉시 멈추므로 이후 유료 호출은 실행되지 않습니다.

**2026-11-02 실행 기록:** 저자가 한 대의 컴퓨터에서 이 노트북을 처음부터 끝까지 한 번 실행했을 때 Claude 호출은 2회(4절의 모델 직접 호출 1회, Deep Agents 한 턴 1회), 판정 API(Decision API) 호출은 1회(2절)였고, 전체 실행 시간은 약 16초였습니다(로컬 임베딩·재순위 모델 파일이 이미 내려받아진 상태). 위 안내 규모(총 3회 내외)와 같은 수준이지만, 한 번의 실행이 한 대의 컴퓨터에서 나온 기록이므로 네트워크와 모델 응답 속도에 따라 시간은 달라질 수 있습니다.


## 1. 설치와 폴더 구조 (2분)

패키지 설치는 **저장소 루트 폴더에서 딱 한 번만** 실행합니다. 루트의 `pyproject.toml`이 전체 과정을 묶는 `uv workspace`로 설정되어 있어서, `uv.lock`과 가상환경(`.venv`)도 루트에 하나만 존재합니다.

```bash
uv sync --locked            # 저장소 루트에서 한 번만 실행합니다
cd Day02_new
uv run jupyter lab          # 주피터 실행은 반드시 Day02_new 폴더에서 시작합니다
```

> **주의:** `Day02_new/` 폴더 안에서 `uv sync`를 실행하지 마세요. 이 폴더에 당장 필요하지 않은 다른 일차(Day)의 패키지가 가상환경에서 지워져 다른 실습이 깨질 수 있습니다. 만약 실수로 실행했다면 저장소 루트로 이동해 `uv sync --locked`를 다시 실행하면 복구됩니다(루트 `AGENTS.md`의 실행 규칙 참고).

먼저 아래 코드 셀을 실행해 현재 주피터 커널(kernel)이 루트의 `.venv` 가상환경을 제대로 바라보고 있는지 확인해 봅시다.


In [ ]:
import sys
from importlib.metadata import version

# [1단계] 지금 노트북을 돌리는 파이썬과 deepagents 버전을 읽습니다
python_version = sys.version.split()[0]
deepagents_version = version("deepagents")

# [2단계] 커널이 루트 .venv 가상환경을 쓰는지 경로 끝부분으로 판별합니다
venv_path = sys.prefix
uses_root_venv = venv_path.endswith(".venv")

# [3단계] 결과를 한국어 라벨과 함께 출력합니다
print("Python 버전          :", python_version)
print("deepagents 버전      :", deepagents_version)
print("가상환경 경로(prefix):", venv_path)
print("루트 .venv 사용 여부 :", uses_root_venv)

**관찰할 포인트:**
- `루트 .venv 사용 여부`가 `True`로 출력되었다면 올바른 가상환경에서 실행 중입니다.

이어서 아래 셀을 실행해 작업 기준 폴더(`ROOT`) 아래에 실습용 Skill 폴더(`skills/`)와 사내 문서 폴더(`data/business/`)가 제대로 보이는지 확인해 봅시다.


In [ ]:
from rag_lab.settings import ROOT

# [1단계] skills/ 아래에서 SKILL.md 파일을 가진 폴더만 Skill로 인정해 이름을 모읍니다
skills_dir = ROOT / "skills"
skill_folders = []
for path in skills_dir.iterdir():
    has_skill_file = (path / "SKILL.md").exists()
    if has_skill_file:
        skill_folders.append(path.name)
skill_folders = sorted(skill_folders)

# [2단계] data/business/ 아래의 사내 문서 파일 이름을 모읍니다
documents_dir = ROOT / "data" / "business"
documents = sorted(path.name for path in documents_dir.iterdir())

# [3단계] 결과를 출력합니다
print("작업 기준 폴더(ROOT) :", ROOT.name)
print("제공된 Skill 목록    :", skill_folders)
print("사내 문서 파일 수    :", len(documents), "개 파일")

**관찰할 포인트:**
- `from rag_lab...`처럼 별도 경로 설정 없이 바로 불러올 수 있는 이유는 이 노트북 파일이 `rag_lab/` 폴더와 나란히 놓여 있고, 커널의 현재 작업 디렉터리(working directory)가 `Day02_new/`이기 때문입니다. 만약 상위 폴더 등 다른 위치에서 커널을 띄우면 `rag_lab`을 찾지 못해 `ModuleNotFoundError`가 발생합니다(이 현상은 마지막 `## 직접 해보기` 2번에서 직접 확인해 봅니다).

---

## 2. 키 상태와 `.env` 설정 (3분)

API 키는 반드시 `Day02_new/.env` 파일 안에만 작성합니다. 예제 파일을 복사해 직접 만들어 주세요.

```bash
cp Day02_new/.env.example Day02_new/.env   # 복사한 뒤 편집기로 열어 본인의 API 키를 채웁니다
```

- **보안 원칙:** 키 원본 값은 노트북 화면, 실행 로그, 결과 JSON 파일 어디에도 출력하지 않습니다. `key_status()` 함수는 오직 `설정됨` 또는 `미설정` 여부만 표시합니다.
- **격리된 환경 변수:** 다른 Day 폴더의 `.env` 파일이나 터미널 셸(shell)에 미리 `export`해 둔 환경 변수는 일부러 읽지 않습니다. 의도치 않게 다른 실습용 키로 과금이 일어나는 사고를 막기 위해서입니다.
- **커널 재시작:** `.env` 파일을 수정하고 저장한 뒤에는 **주피터 커널을 재시작(Restart Kernel)** 해야 변경된 키가 반영됩니다.
- **실제 모델 전용:** 이 과정은 가짜(mock) 모델로 우회하는 경로가 없습니다. 키가 비어 있으면 아래 셀에서 곧바로 멈춥니다.


In [ ]:
from rag_lab.settings import key_status, require_key

# [1단계] 현재 .env 파일의 설정 상태를 확인합니다 (키 값 자체는 출력하지 않고 설정됨/미설정만 보여 줍니다)
statuses = key_status()
for name, value in statuses.items():
    print(f"{name:<24} {value}")

# [2단계] ANTHROPIC_API_KEY가 없으면 한국어 안내 메시지와 함께 RuntimeError를 일으켜 여기서 멈춥니다
settings = require_key()

# [3단계] 이후 셀들이 쓸 기본 모델 이름을 확인합니다 (settings는 4절의 모델 호출 셀에서도 다시 씁니다)
print("기본 사용 모델 :", settings.model)

### 판정 API(Decision API) 키 확인 (필수)

이 과정에서는 답변을 생성하는 Claude(`ANTHROPIC_API_KEY`) 외에 **판정 API(Decision API) 키가 하나 더 필요합니다.** `TYPESAFE_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY`(또는 Azure OpenAI 설정) 중 본인이 사용할 공급자의 키 하나를 `Day02_new/.env`에 **직접** 채워 주세요. 건너뛰는 경로가 없으므로 키가 비어 있으면 아래 셀에서 바로 멈춥니다. 여러 공급자의 키를 함께 적어 둔 경우에는 `DAY02_DECISION_BACKEND` 값으로 공급자를 지정할 수 있고, 비워 두면 위 순서대로 키가 설정된 첫 번째 공급자를 자동 선택합니다.

- **왜 판정 API를 따로 쓰나요?** 일반 채팅 모델은 자유로운 문장을 생성하지만, 판정 API는 참/거짓(`Noul`)처럼 정해진 기준에 대해 **0과 1 사이의 확률 숫자**를 돌려줍니다. 덕분에 `확률 >= 0.5일 때만 질문을 분해한다`처럼 코드로 명확한 문턱값(gate)을 걸어 실행 흐름을 제어할 수 있습니다.
- **데이터 전송 주의:** 04~06번 노트북의 게이트 셀에서는 질문 문장과 검색된 근거 텍스트가 선택한 판정 API 공급자로 전송됩니다. 외부로 유출되면 안 되는 민감한 사내 기밀이나 개인정보는 입력하지 마세요.

아래 셀은 간단한 참/거짓 판정 문항(`Noul`) 하나를 만들어 `decider.judge(...)`를 실제로 한 번 호출해 봅니다.


In [ ]:
import time
from rag_lab.settings import announce
from rag_lab.decisions import Noul, make_decider

# [1단계] 유료 API 호출 직전에 예상 호출 규모를 한 줄로 알립니다
announce("판정 API 동작 확인", model_calls=1)

# [2단계] 판정 API 클라이언트를 만듭니다 (키가 없으면 한국어 안내와 함께 RuntimeError로 멈춥니다)
decider = make_decider()

# [3단계] 참(true)/거짓(false) 기준이 명확한 판정 문항(Noul)을 정의합니다
question = Noul(
    name="is_korean",
    instructions="주어진 문장이 한국어로 작성되었는지 판정합니다.",
    true="문장이 한국어로 작성되었다.",
    false="문장이 한국어로 작성되지 않았다.",
)

# [4단계] 판정할 문장을 정하고, 판정 API를 실제로 한 번 호출하면서 걸린 시간을 잽니다
sentence = "서비스 크레딧은 다음 달 말일까지 신청합니다."
started = time.perf_counter()
judgment = decider.judge(sentence, [question])
elapsed = time.perf_counter() - started

# [5단계] 결과에서 문항 이름(is_korean)으로 판정을 꺼내 참일 확률을 읽습니다
verdict = judgment.verdicts["is_korean"]
probability = verdict.probability

print("선택된 공급자(backend) :", judgment.backend)
print("판정 모델(model)       :", judgment.model)
print("참일 확률(probability) :", probability)
print(f"소요 시간              : {elapsed:.2f}초")

# [6단계] 확률이 0~1 사이의 숫자인지 검사합니다 (아니면 연결이 비정상이라는 뜻입니다)
assert probability is not None and 0 <= probability <= 1, "확률은 0~1 사이의 숫자여야 합니다"

**관찰할 포인트:**
- `참일 확률(probability)`에 0과 1 사이의 숫자가 출력되고 `assert` 검사를 통과했다면 판정 API 연결이 정상적으로 준비된 것입니다.

---

## 3. 토크나이저, 로컬 임베딩 모델, 재순위 모델 점검 (2분)

이제 검색(Retrieval)에 사용할 세 가지 로컬 도구가 내 컴퓨터에서 잘 돌아가는지 차례대로 확인합니다.

1. **한국어 형태소 분석기(Kiwi):** 단어에 붙은 조사나 어미를 떼어 내어 검색용 토큰으로 쪼개 줍니다. 처음 실행할 때는 언어 사전을 메모리에 올리느라 1초 안팎이 걸립니다(컴퓨터에 따라 다를 수 있습니다).


In [ ]:
import time
from rag_lab.retrieval import tokenize

# [1단계] 형태소 분석기(Kiwi)로 문장을 쪼개고, 첫 실행의 사전 적재 시간까지 함께 잽니다
text = "서비스 크레딧 신청 기한"
started = time.perf_counter()
tokens = tokenize(text)
elapsed = time.perf_counter() - started

# [2단계] 추출된 토큰과 소요 시간을 출력합니다
print("추출된 형태소 토큰 :", tokens)
print(f"첫 실행 소요 시간  : {elapsed:.2f}초")

**관찰할 포인트:**
- 문장에서 핵심 명사인 `[서비스, 크레딧, 신청, 기한]` 네 개가 깔끔하게 추출된 것을 볼 수 있습니다.

2. **로컬 임베딩 모델(`intfloat/multilingual-e5-small`):** Anthropic은 별도의 임베딩 API를 제공하지 않기 때문에, 의미 기반 검색(dense retrieval)에 쓸 벡터는 내 컴퓨터에서 도는 로컬 모델로 만듭니다. 처음 실행할 때 약 470MB 크기의 모델 파일을 내려받습니다.


In [ ]:
from rag_lab.settings import make_embeddings

# [1단계] 로컬 임베딩 모델을 불러오고(처음에는 내려받기 포함) 문장 하나를 벡터로 바꿉니다
text = "서비스 크레딧 신청 기한"
started = time.perf_counter()
embeddings = make_embeddings()
vector = embeddings.embed_query(text)
elapsed = time.perf_counter() - started

# [2단계] 벡터의 앞 5개 값만 소수점 넷째 자리로 반올림해 예시로 보여 줍니다
first_five = []
for value in vector[:5]:
    first_five.append(round(value, 4))

print("임베딩 벡터 차원 수 :", len(vector))
print("벡터 앞 5개 값 예시 :", first_five)
print(f"모델 적재 포함 시간 : {elapsed:.2f}초")

**관찰할 포인트:**
- 문장 하나가 384개의 실수로 이루어진 고정 길이 벡터(`임베딩 벡터 차원 수 : 384`)로 변환되었습니다.

3. **로컬 재순위 모델(`BAAI/bge-reranker-v2-m3`, 필수):** 04~06번 노트북에서 1차로 검색된 후보 문서 조각들을 질문과 직접 짝지어 정밀하게 다시 줄 세우는 교차 인코더(cross-encoder) 모델입니다. 처음 실행할 때 **약 2.1GB**의 모델 파일을 내려받으므로 네트워크가 원활한 환경에서 수업 전에 미리 실행해 두세요. 한 번 내려받은 뒤에는 내 컴퓨터에서 로컬로 실행되므로 API 호출 비용이 들지 않습니다.


In [ ]:
from rag_lab.settings import make_reranker

# [1단계] 질문 하나와, 관련 있는 문장 / 무관한 문장을 준비합니다
query = "서비스 크레딧 신청 기한"
relevant_doc = "서비스 크레딧은 장애가 발생한 달의 다음 달 말일까지 신청해야 합니다."
irrelevant_doc = "사내 식당은 평일 오전 11시 30분부터 오후 1시까지 운영합니다."
candidates = [relevant_doc, irrelevant_doc]

# [2단계] 재순위 모델을 불러오고(처음에는 내려받기 포함) 질문-문장 쌍마다 점수를 매깁니다
started = time.perf_counter()
reranker = make_reranker()
scores = reranker.score(query, candidates)
elapsed = time.perf_counter() - started

# [3단계] scores[0]은 관련 문장, scores[1]은 무관한 문장의 점수입니다
relevant_score = scores[0]
irrelevant_score = scores[1]
print(f"관련 있는 문장 점수 : {relevant_score:.2f}")
print(f"무관한 문장 점수    : {irrelevant_score:.2f}")
print(f"모델 적재 포함 시간 : {elapsed:.2f}초")

# [4단계] 관련 있는 문장의 점수가 더 높아야 재순위 모델이 정상 동작하는 것입니다
assert relevant_score > irrelevant_score, "질문과 관련 있는 문장의 점수가 더 높아야 합니다"

### 방금 실행한 결과가 보장하는 것과 보장하지 않는 것

세 가지 로컬 도구가 모두 정상 실행되었다고 해서 곧바로 RAG 시스템의 품질까지 보장되는 것은 아닙니다. **환경이 정상 동작한다는 사실**과 **이후 수업에서 설계하고 검증해야 할 품질 문제**를 명확히 구분해 두어야 합니다.

| 방금 확인한 사실 (환경 정상 여부) | 아직 확인하지 않은 사실 (이후 수업에서 다룰 품질 문제) |
|---|---|
| 필요한 파이썬 패키지가 설치되어 정상적으로 `import` 됩니다 | 검색기가 실제 사내 문서에서 정답 문서를 빠짐없이 찾아냅니다 |
| Kiwi가 한국어 문장을 형태소 토큰으로 쪼갭니다 | 임베딩 유사도만으로 까다로운 질문을 구분해 낼 수 있습니다 |
| 로컬 임베딩 모델이 메모리에 올라와 고정 길이 벡터를 만듭니다 | LLM이 상황에 맞는 올바른 도구를 스스로 선택해 호출합니다 |
| 재순위 모델이 질문-문장 쌍에 대해 점수를 매깁니다 | 재순위를 거치면 언제나 검색 순위가 더 좋아집니다 |

---

## 4. 실제 모델 호출과 Deep Agents 한 턴 실행 (2분)

유료 모델을 호출하는 모든 셀은 실행 직전에 `announce(...)` 함수로 예상 호출 규모를 한 줄로 먼저 알립니다. 먼저 아래 셀을 실행해 Claude 모델을 직접 한 번 호출해 봅시다(뒤이어 나오는 다른 셀들은 이 결과 변수에 의존하지 않으므로 질문을 자유롭게 바꿔 보셔도 됩니다).


In [ ]:
from rag_lab.settings import announce, make_model

# [1단계] 유료 호출 직전에 예상 호출 규모를 알리고, 2절에서 확인한 settings로 모델을 만들어 한 번 호출합니다
announce("모델 직접 호출 확인", model_calls=1)
model = make_model(settings)
reply = model.invoke("한 단어로 답하세요: 준비")

# [2단계] reply.content가 블록(dict) 리스트인지 단일 문자열인지 확인해 블록 종류를 살펴봅니다
block_types = []
if isinstance(reply.content, list):
    for block in reply.content:
        if isinstance(block, dict):
            block_types.append(block["type"])
        else:
            block_types.append("text")
else:
    block_types.append("text")

# [3단계] 최종 답변 문자열(.text)과 내용 블록 종류를 출력합니다
print("모델 응답(.text) :", reply.text)
print("내용 블록 종류   :", block_types)

**관찰할 포인트:**
- `announce`가 호출 규모 안내문을 먼저 출력했는지 확인하세요.
- `reply.text`로 최종 답변 문자열을 꺼낼 수 있는지 확인하세요. 원본 `reply.content`가 문자열인지 블록 리스트인지, 블록 리스트라면 `내용 블록 종류`에 `text` 외에 모델의 확장 추론 과정인 `thinking` 블록이 함께 있는지는 모델 설정과 응답에 따라 달라질 수 있으니, 출력된 블록 종류를 직접 읽어 보세요. 어느 쪽이든 최종 답변은 `reply.text`로 읽습니다.

이제 방금 확인한 모델을 `create_deep_agent`에 넣어 에이전트를 한 턴 실행해 봅시다. 실행 결과의 `messages` 리스트에 사용자 질문(`HumanMessage`)과 모델 응답(`AIMessage`)이 순서대로 담겨 있는지 `전체 메시지 수`와 각 줄의 클래스 이름으로 확인해 보세요(도구를 쓰지 않은 한 턴이라면 보통 두 개입니다).


In [ ]:
from deepagents import create_deep_agent

# [1단계] 유료 호출 직전에 예상 호출 규모를 알립니다 (announce는 위 셀에서 불러온 것을 그대로 씁니다)
announce("Deep Agents 한 턴 실행", model_calls=1)

# [2단계] 모델을 넣어 Deep Agent를 만들고, 사용자 질문 한 줄로 한 턴을 실행합니다
agent = create_deep_agent(model=make_model(settings))
user_message = {"role": "user", "content": "한 단어로 답하세요: 준비"}
result = agent.invoke({"messages": [user_message]})

# [3단계] 주고받은 메시지를 순서대로 출력합니다 (HumanMessage, AIMessage 등)
messages = result["messages"]
print("전체 메시지 수 :", len(messages))
for idx, msg in enumerate(messages):
    print(f"  [{idx}] {type(msg).__name__}: {msg.text}")

### 오류가 발생했을 때 점검할 곳

만약 위 셀들에서 오류가 발생했다면 아래 표를 보고 원인을 먼저 점검해 주세요.

| 증상 | 먼저 확인할 사항 |
|---|---|
| 키 안내 메시지와 함께 `RuntimeError` 발생 | `Day02_new/.env` 파일이 존재하는지, 키를 입력한 뒤 커널을 재시작(Restart)했는지 확인합니다 |
| 판정 API 셀에서 `RuntimeError` 발생 | `.env`에 판정 API 키(`TYPESAFE_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` 중 하나)를 채웠는지, `DAY02_DECISION_BACKEND` 철자가 맞는지 확인합니다 |
| `401 Unauthorized` 오류 | API 키 문자열에 오타나 공백이 없는지, 만료되지 않았는지 각 공급자 콘솔에서 확인합니다 (노트북에 키 값을 직접 출력하지 마세요) |
| 요청 시간 초과 (Timeout) | 사내망/방화벽 등 네트워크 연결 상태를 점검합니다 (모델 호출 제한 시간은 120초입니다) |
| `ModuleNotFoundError` | 저장소 루트에서 `uv sync --locked`를 실행했는지, 주피터 커널이 루트 `.venv`를 바라보는지, 현재 작업 폴더가 `Day02_new/`인지 확인합니다 |

---

## 5. 이후 수업 시간표와 비용 규칙 (1분)

| 노트북 | 주제 | 권장 시간 |
|---|---|---:|
| `01-Agentic-AI-Basics.ipynb` | Agentic AI 기초: 단일 호출, 고정 workflow, Agent 비교 | 60분 |
| `02-Function-Calling-and-Tools.ipynb` | Function Calling과 도구 설계 | 60분 |
| `03-Agent-Skills.ipynb` | Agent Skills: 필요할 때만 읽는 절차 문서 | 60분 |
| `04-Retrieval-Design.ipynb` | 검색(Retrieval) 설계와 측정 | 60분 |
| `05-Production-Agentic-RAG.ipynb` | 운영형 Agentic RAG의 9가지 설계 결정과 게이트 | 60분 |
| `06-Lab-DeepAgents-RAG.ipynb` | DeepAgents RAG 전체 조립 및 통제 실험 실습 | 60분 |

**비용 고지 규칙:** 실제 유료 모델을 호출하는 모든 셀은 실행 직전에 예상 호출 규모를 한 줄로 먼저 알립니다(`이 실행은 모델 호출 약 N회...`). 필수 키(`ANTHROPIC_API_KEY`와 판정 API 키)가 없으면 어떤 노트북도 실행되지 않으며, 키를 설정하고 실행하면 본인의 API 계정 크레딧이 사용됩니다.

## 직접 해보기

1. `.env` 파일 이름을 잠시 바꾸거나 키를 비운 상태에서 2절 셀을 실행해 어디서 어떤 안내 메시지와 함께 멈추는지 확인한 뒤, 다시 `.env`를 복구하고 커널을 재시작해 `key_status()` 출력을 비교해 보세요.
2. 터미널에서 `Day02_new/` 바깥 폴더(예: 저장소 루트)에서 파이썬을 실행해 `import rag_lab`을 시도해 보고, 왜 실패하는지 한 줄로 정리해 보세요.
3. 4절 Deep Agents 셀의 질문 내용을 자유롭게 바꿔 보고, `result["messages"]`의 메시지 개수와 마지막 응답이 어떻게 달라지는지 확인해 보세요.

## 확인 질문

1. 4절의 모델 호출 셀이 에러 없이 실행되었다고 해서, 곧바로 우리 RAG 시스템의 검색이나 답변 품질이 좋다고 말할 수 없는 이유는 무엇인가요?
2. 터미널 셸에 `export ANTHROPIC_API_KEY=...`로 키를 등록해 두어도 `Day02_new/.env`에 적지 않으면 `key_status()`가 `미설정`이라고 표시하는 이유는 무엇이며, 이 설계는 어떤 실수를 막아 주나요?
3. 저장소 루트가 아니라 `Day02_new/` 폴더 안에서 `uv sync`를 실행하면 다른 일차(Day) 실습 환경에 어떤 문제가 생기나요?

## 정리

- 패키지 설치는 저장소 루트에서 `uv sync --locked`로 한 번만 하고, 주피터 실행은 `Day02_new/` 폴더에서 합니다.
- 두 종류의 필수 키(`ANTHROPIC_API_KEY`와 판정 API 키)는 `Day02_new/.env`에 직접 작성하며, 화면에는 `설정됨`/`미설정` 상태만 출력합니다.
- 이 과정은 실제 모델만 사용합니다. 키가 없으면 `require_key()`와 `make_decider()`에서 즉시 멈추며, 임베딩과 재순위는 내 컴퓨터의 로컬 모델이 처리합니다.
- 다음 시간에는 `01-Agentic-AI-Basics.ipynb`에서 일반 LLM 호출과 Agent의 근본적인 차이를 직접 실행하며 살펴봅니다.
